# Tutorial 2: Time-series data quality and benchmark forecasting

## 1. Overview and learning outcomes

This tutorial accompanies Lecture 2 of **BENV0169 Data Analytics for Sustainable Buildings**. We use hourly aggregate electricity demand to connect time-index quality, missing-data treatment, rolling- and fixed-origin forecasts, and forecast evaluation.

By the end of the tutorial, you should be able to:

- inspect a time index and distinguish absent timestamps from explicit `NaN` values;
- compare imputation methods using artificially hidden observations with known ground truth;
- construct day-ahead, rolling-origin and fixed week-ahead benchmark forecasts;
- calculate RMSE, MAPE and MASE with appropriate input checks;
- compare forecasts without leaking future information.

### Indicative schedule

| Activity | Time |
|---|---:|
| Setup and time-index checks | 10 minutes |
| Imputation experiment | 25 minutes |
| Benchmark methods and metrics | 20 minutes |
| Day-ahead, rolling and fixed-week experiments | 30 minutes |
| Interpretation | 5 minutes |

## 2. Setup and data loading

The dataset comes from the load-forecasting track of GEFCom2012. Columns `Z1` to `Z20` represent twenty zones and `Z21` is their aggregate. Demand is measured in MW.

**Reference**

Hong, T., Pinson, P. and Fan, S. (2014) ‘Global Energy Forecasting Competition 2012’, *International Journal of Forecasting*, 30(2), pp. 357–363. [https://doi.org/10.1016/j.ijforecast.2013.07.001](https://doi.org/10.1016/j.ijforecast.2013.07.001)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)


In [ ]:
DATA_PATH = Path("data") / "tutorial_02" / "load_data_full.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Launch Jupyter from the repository root."
    )

load_data = pd.read_csv(DATA_PATH, index_col=0, parse_dates=[0])
load_data.index.name = "timestamp"
load_data = load_data.sort_index()

if not isinstance(load_data.index, pd.DatetimeIndex):
    raise TypeError("The first CSV column must parse as a DatetimeIndex.")
if "Z21" not in load_data.columns:
    raise KeyError("Expected the aggregate load column 'Z21'.")

print(f"Loaded {len(load_data):,} rows and {load_data.shape[1]} demand columns.")
load_data.head(3)


## 3. Time-index quality

### Worked example: inspect the time index

The following cells illustrate the checks that should precede time-series analysis. They identify the coverage and nominal sampling interval, verify timestamp uniqueness, find timestamps absent from the expected hourly index, and count explicit `NaN` values in stored rows.

Run the supplied code and inspect the summary and missing-block table. Notice that missing timestamps are identified **before** the index is regularised. No code completion is required in this section.


In [ ]:
def summarise_time_index(data, expected_frequency="h"):
    '''Return key time-index and explicit-missingness checks.'''
    if not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("data must have a DatetimeIndex.")
    if data.empty:
        raise ValueError("data must contain at least one row.")

    time_differences = data.index.to_series().diff().dropna()
    nominal_interval = time_differences.mode().iloc[0]
    expected_index = pd.date_range(
        start=data.index.min(),
        end=data.index.max(),
        freq=expected_frequency,
    )
    missing_timestamps = expected_index.difference(data.index)

    return {
        "start": data.index.min(),
        "end": data.index.max(),
        "nominal_interval": nominal_interval,
        "duplicate_timestamps": int(data.index.duplicated().sum()),
        "missing_timestamps": missing_timestamps,
        "rows_with_nan": int(data.isna().any(axis=1).sum()),
        "nan_cells": int(data.isna().sum().sum()),
    }


In [ ]:
time_quality = summarise_time_index(load_data)

print("Data-quality summary")
print(f"  Coverage: {time_quality['start']} to {time_quality['end']}")
print(f"  Nominal interval: {time_quality['nominal_interval']}")
print(f"  Duplicate timestamps: {time_quality['duplicate_timestamps']:,}")
print(f"  Missing timestamps: {len(time_quality['missing_timestamps']):,}")
print(f"  Stored rows containing NaN: {time_quality['rows_with_nan']:,}")
print(f"  Explicit NaN cells: {time_quality['nan_cells']:,}")


In [ ]:
def consecutive_timestamp_blocks(timestamps, frequency="h"):
    '''Summarise a DatetimeIndex as consecutive blocks.'''
    timestamps = pd.DatetimeIndex(timestamps).sort_values()
    if timestamps.empty:
        return pd.DataFrame(columns=["start", "end", "hours"])

    step = pd.to_timedelta(1, unit=frequency)
    block_starts = [timestamps[0]]
    block_ends = []

    for previous, current in zip(timestamps[:-1], timestamps[1:]):
        if current - previous != step:
            block_ends.append(previous)
            block_starts.append(current)
    block_ends.append(timestamps[-1])

    blocks = pd.DataFrame({"start": block_starts, "end": block_ends})
    blocks["hours"] = (
        (blocks["end"] - blocks["start"]) / pd.Timedelta(hours=1) + 1
    ).astype(int)
    return blocks


missing_blocks = consecutive_timestamp_blocks(time_quality["missing_timestamps"])
missing_blocks


In [ ]:
if time_quality["duplicate_timestamps"] != 0:
    raise ValueError("Resolve duplicate timestamps before reindexing.")

full_hourly_index = pd.date_range(
    start=load_data.index.min(),
    end=load_data.index.max(),
    freq="h",
    name="timestamp",
)
hourly_load_data = load_data.reindex(full_hourly_index)

newly_created_missing_rows = hourly_load_data.index.difference(load_data.index)
print(f"Rows after hourly reindexing: {len(hourly_load_data):,}")
print(f"Rows inserted by reindexing: {len(newly_created_missing_rows):,}")


**Interpretation.** An absent timestamp has no stored row, so it only becomes visible when the observed index is compared with the expected hourly index. An explicit `NaN` occupies an existing row but lacks a value in one or more columns. Reindexing converts absent timestamps into new rows containing `NaN`, so the two cases must be counted before regularisation.


### Seasonal patterns in a complete year

The 2007 portion contains a complete hourly index. The first plot shows two weeks of `Z21`. The profile plots aggregate 2007 by hour of day and hour of week. Use them to identify the daily period $m=24$ and weekly period $m=168$.


In [ ]:
aggregate_load_2007 = hourly_load_data.loc["2007", "Z21"].copy()
expected_2007_index = pd.date_range(
    "2007-01-01 00:00", "2007-12-31 23:00", freq="h", name="timestamp"
)

if not aggregate_load_2007.index.equals(expected_2007_index):
    raise ValueError("The selected 2007 index is not complete and hourly.")
if aggregate_load_2007.isna().any():
    raise ValueError("The selected 2007 aggregate series contains missing values.")

two_weeks = aggregate_load_2007.loc["2007-10-01":"2007-10-14 23:00"]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(two_weeks.index, two_weeks, color="tab:blue", linewidth=1.2)
ax.set(
    title="Aggregate hourly demand over two weeks",
    xlabel="Timestamp",
    ylabel="Demand (MW)",
)
ax.grid(alpha=0.25)
plt.show()

mean_by_hour = aggregate_load_2007.groupby(aggregate_load_2007.index.hour).mean()
hour_of_week = aggregate_load_2007.index.dayofweek * 24 + aggregate_load_2007.index.hour
mean_by_hour_of_week = aggregate_load_2007.groupby(hour_of_week).mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mean_by_hour.index, mean_by_hour, marker="o", markersize=3)
axes[0].set(
    title="Mean demand by hour of day",
    xlabel="Hour of day",
    ylabel="Mean demand (MW)",
)
axes[0].set_xticks(range(0, 24, 3))
axes[0].grid(alpha=0.25)

axes[1].plot(mean_by_hour_of_week.index, mean_by_hour_of_week)
axes[1].set(
    title="Mean demand by hour of week",
    xlabel="Hour of week (Monday 00:00 = 0)",
    ylabel="Mean demand (MW)",
)
axes[1].set_xticks(range(0, 169, 24))
axes[1].grid(alpha=0.25)
plt.show()


## 4. Imputation experiment

The long gaps in 2005 and 2006 have no observed ground truth, so they cannot tell us which imputation method reconstructs the missing demand accurately. Instead, we take an untouched, complete interval from 2007 and hide known values deterministically.

The artificial mask contains:

- one isolated missing hour;
- one six-hour gap;
- one 24-hour gap.

We retain the original values as `imputation_truth`. Every method must receive `corrupted_load`, never the ground-truth values.


In [ ]:
imputation_truth = aggregate_load_2007.loc[
    "2007-09-23 00:00":"2007-10-28 23:00"
].copy()

artificial_gaps = {
    "Isolated hour": pd.date_range("2007-10-02 14:00", periods=1, freq="h"),
    "Six-hour gap": pd.date_range("2007-10-10 09:00", periods=6, freq="h"),
    "24-hour gap": pd.date_range("2007-10-20 00:00", periods=24, freq="h"),
}
masked_timestamps = pd.DatetimeIndex(
    sorted(timestamp for gap in artificial_gaps.values() for timestamp in gap)
)

if not masked_timestamps.isin(imputation_truth.index).all():
    raise ValueError("Every masked timestamp must lie inside the complete interval.")
if imputation_truth.loc[masked_timestamps].isna().any():
    raise ValueError("Ground truth must be observed at every masked timestamp.")

corrupted_load = imputation_truth.copy()
corrupted_load.loc[masked_timestamps] = np.nan

print(f"Artificially masked observations: {corrupted_load.isna().sum()}")
for gap_name, gap_index in artificial_gaps.items():
    print(f"  {gap_name}: {gap_index[0]} to {gap_index[-1]} ({len(gap_index)} hour(s))")


### Inspect the artificial gaps

Before selecting an imputation method, examine the corrupted series around each gap. Consider how rapidly demand changes, whether a repeated daily pattern is visible, and how the gap length affects the information available for reconstruction.

The shaded regions identify the observations that require imputation. Their true values remain hidden at this stage.


In [ ]:
fig, axes = plt.subplots(
    len(artificial_gaps),
    1,
    figsize=(11, 9),
    sharey=True,
)

for ax, (gap_name, gap_index) in zip(axes, artificial_gaps.items()):
    window_start = gap_index[0] - pd.Timedelta(hours=24)
    window_end = gap_index[-1] + pd.Timedelta(hours=24)
    visible_data = corrupted_load.loc[window_start:window_end]

    ax.plot(
        visible_data.index,
        visible_data,
        color="tab:blue",
        marker="o",
        markersize=2.5,
        linewidth=1.2,
        label="Observed demand",
    )
    ax.axvspan(
        gap_index[0],
        gap_index[-1] + pd.Timedelta(hours=1),
        color="tab:red",
        alpha=0.12,
        label="Artificial gap",
    )
    ax.set_title(gap_name)
    ax.set_ylabel("Demand (MW)")
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
axes[-1].set_xlabel("Timestamp")
fig.suptitle("Corrupted demand series before imputation", fontsize=13)
plt.show()


### Task 1: implement the imputation methods

Implement four simple comparisons:

1. forward filling;
2. replacement by the mean of the observed values in the selected interval;
3. time-based linear interpolation;
4. seasonal replacement from the same hour in the preceding week, $m=168$.

Return a complete `Series` with the same index. Preserve every non-missing observation. Do not read from `imputation_truth` inside any imputation function.


In [ ]:
def forward_fill_imputation(series):
    '''Fill each gap using the last observed value.'''
    # TODO: apply forward filling and check that no NaN remains.
    raise NotImplementedError("Implement forward_fill_imputation().")


def observed_mean_imputation(series):
    '''Fill gaps with the mean of all observed values in this interval.'''
    # TODO: calculate the mean from series after ignoring NaN values.
    raise NotImplementedError("Implement observed_mean_imputation().")


def linear_interpolation_imputation(series):
    '''Fill internal gaps by linear interpolation in time.'''
    # TODO: use Series.interpolate with the DatetimeIndex.
    raise NotImplementedError("Implement linear_interpolation_imputation().")


def previous_week_imputation(series, period=168):
    '''Replace each missing value with the observation one week earlier.'''
    # TODO: for each missing timestamp, read from timestamp - period hours.
    # Use only values in series, not imputation_truth.
    raise NotImplementedError("Implement previous_week_imputation().")


In [ ]:
imputed_series = {
    "Forward fill": forward_fill_imputation(corrupted_load),
    "Observed mean": observed_mean_imputation(corrupted_load),
    "Linear interpolation": linear_interpolation_imputation(corrupted_load),
    "Previous week": previous_week_imputation(corrupted_load, period=168),
}

observed_mask = corrupted_load.notna()
for method_name, reconstructed in imputed_series.items():
    if not reconstructed.index.equals(corrupted_load.index):
        raise ValueError(f"{method_name} changed the index.")
    if not reconstructed.loc[observed_mask].equals(corrupted_load.loc[observed_mask]):
        raise ValueError(f"{method_name} changed observations that were not masked.")


### Task 2: evaluate reconstruction accuracy

Complete `score_imputations`. Calculate RMSE and MAE **only at the artificially masked timestamps**. Lower values indicate a closer reconstruction of the hidden observations.


In [ ]:
def score_imputations(truth, missing_index, reconstructions):
    '''Return RMSE and MAE at deliberately masked timestamps only.'''
    # TODO: subset truth and each reconstruction with missing_index.
    # Calculate error as actual - reconstructed and return a DataFrame
    # with methods as rows and RMSE (MW), MAE (MW) as columns.
    raise NotImplementedError("Complete score_imputations().")


In [ ]:
imputation_scores = score_imputations(
    imputation_truth,
    masked_timestamps,
    imputed_series,
)
imputation_scores.round(2)


### Visual comparison

The metrics summarise all 31 hidden observations. The plots reveal how each method behaves for different gap lengths and whether it attenuates peaks or creates artificial flat sections.


In [ ]:
def plot_imputation_window(gap_name, gap_index, truth, corrupted, reconstructions):
    '''Plot 24 hours of context on either side of one artificial gap.'''
    window_start = gap_index[0] - pd.Timedelta(hours=24)
    window_end = gap_index[-1] + pd.Timedelta(hours=24)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(
        truth.loc[window_start:window_end],
        color="black",
        linewidth=2.2,
        label="True demand",
    )
    ax.plot(
        corrupted.loc[window_start:window_end],
        color="0.65",
        marker="o",
        markersize=2.5,
        linewidth=1,
        label="Corrupted series",
    )
    for method_name, reconstructed in reconstructions.items():
        ax.plot(
            reconstructed.loc[window_start:window_end],
            linewidth=1.2,
            alpha=0.9,
            label=method_name,
        )

    ax.axvspan(
            gap_index[0],
            gap_index[-1] + pd.Timedelta(hours=1),
            color="tab:red",
            alpha=0.08,
        )
    ax.set(title=gap_name, xlabel="Timestamp", ylabel="Demand (MW)")
    ax.grid(alpha=0.2)
    ax.legend(ncol=2, fontsize=8)
    plt.show()


for gap_name, gap_index in artificial_gaps.items():
    plot_imputation_window(
        gap_name,
        gap_index,
        imputation_truth,
        corrupted_load,
        imputed_series,
    )


### Optional extension: average the surrounding weeks

Compare previous-week replacement with the average of the corresponding hours in the preceding and following weeks. This symmetric seasonal average is retrospective because it uses a future observation.


In [ ]:
def surrounding_weeks_average_imputation(series, period=168):
    '''Average matching hours in the preceding and following weeks.'''
    # OPTIONAL TODO: use timestamp - period and timestamp + period.
    raise NotImplementedError(
        "Optional: implement surrounding_weeks_average_imputation()."
    )


## 5. Forecasting experiments

We now return to the original, complete 2007 series. The artificial imputation gaps play no role in forecasting.

The same benchmark methods will be evaluated in three settings:

1. one day-ahead forecast from a single origin, with a fixed horizon of 24 hours;
2. rolling day-ahead forecasts, updated every 24 hours across the complete October-to-December test set and illustrated for one week;
3. one fixed-origin forecast for the full 168-hour illustration week.

Comparing rolling and fixed forecasts over the same illustration week isolates the value of updating the forecast origin as new observations become available.


In [ ]:
# Fixed-origin experiments and the one-week illustration
training_load = aggregate_load_2007.loc[
    "2007-01-01 00:00":"2007-12-16 23:00"
].copy()
test_week = aggregate_load_2007.loc[
    "2007-12-17 00:00":"2007-12-23 23:00"
].copy()
initial_forecast_origin = pd.Timestamp("2007-12-16 23:00")
day_ahead_actual = test_week.iloc[:24].copy()

# Longer test set for the rolling day-ahead experiment
rolling_training_load = aggregate_load_2007.loc[
    "2007-01-01 00:00":"2007-09-30 23:00"
].copy()
rolling_test_load = aggregate_load_2007.loc[
    "2007-10-01 00:00":"2007-12-31 23:00"
].copy()

expected_test_week_index = pd.date_range(
    "2007-12-17 00:00",
    "2007-12-23 23:00",
    freq="h",
    name="timestamp",
)
expected_rolling_test_index = pd.date_range(
    "2007-10-01 00:00",
    "2007-12-31 23:00",
    freq="h",
    name="timestamp",
)

if training_load.index[-1] != initial_forecast_origin:
    raise ValueError("The fixed-origin training set must end at the forecast origin.")
if len(test_week) != 168 or not test_week.index.equals(expected_test_week_index):
    raise ValueError("The illustration week must contain 168 consecutive hours.")
if not rolling_test_load.index.equals(expected_rolling_test_index):
    raise ValueError("The rolling test set must contain every hour in Q4 2007.")
if len(rolling_test_load) % 24 != 0:
    raise ValueError("The rolling test set must contain complete days.")
if len(day_ahead_actual) != 24:
    raise ValueError("The first day-ahead evaluation window must contain 24 hours.")
if any(
    series.isna().any()
    for series in (
        training_load,
        test_week,
        rolling_training_load,
        rolling_test_load,
    )
):
    raise ValueError("All forecasting intervals must be complete.")

print(f"Fixed-origin training observations: {len(training_load):,}")
print(f"Fixed forecast origin: {initial_forecast_origin}")
print(f"Illustration week: {test_week.index[0]} to {test_week.index[-1]}")
print(
    f"Rolling test set: {rolling_test_load.index[0]} to "
    f"{rolling_test_load.index[-1]} ({len(rolling_test_load) // 24} days)"
)


### Fixed and rolling forecast origins

In the rolling experiment, each 24-hour forecast has its own origin. Once one day has passed, its actual observations join the history before the next forecast is issued. The loop repeats this process for every day in the test set. In the fixed week-ahead experiment, the information set remains frozen at the initial origin for all 168 lead times.

> **Operational note.** In real day-ahead electricity-load forecasting, forecasts are commonly issued around 11:00–12:00 on day $D-1$ for delivery on day $D$. This tutorial issues forecasts at 23:00 on day $D-1$. The later origin is a simplification that keeps every forecast window aligned with a complete calendar day.


#### Rolling day-ahead schematic

![Rolling day-ahead forecasting schematic](images/tutorial_02/rolling_day_ahead_schematic_v2.png)

Each square represents one day. **Blue** squares are observations available at that forecast origin; the **orange** square is the next 24-hour forecast. The vertical line marks the origin. Moving down one row advances time by 24 hours: the previous forecast day has been observed, joins the history, and a new day-ahead forecast is issued. Only a short history is drawn; the code below retains the complete expanding history available at each origin.


### Benchmark methods

The historical-mean forecast remains a worked example. The other benchmark functions are completed in Task 3 and reused without alteration in all three forecasting experiments.


In [ ]:
def historical_mean_forecast(train_series, forecast_index):
    '''Repeat the mean of the training observations over the horizon.'''
    if train_series.empty or train_series.isna().any():
        raise ValueError("train_series must be non-empty and contain no missing values.")
    return pd.Series(
        np.repeat(train_series.mean(), len(forecast_index)),
        index=forecast_index,
        dtype=float,
    )


In [ ]:
def _validate_forecast_inputs(train_series, forecast_index):
    '''Apply shared checks for the fixed-origin benchmark functions.'''
    if not isinstance(train_series, pd.Series):
        raise TypeError("train_series must be a pandas Series.")
    if not isinstance(train_series.index, pd.DatetimeIndex):
        raise TypeError("train_series must have a DatetimeIndex.")
    if train_series.empty or train_series.isna().any():
        raise ValueError("train_series must be non-empty and contain no missing values.")
    if not train_series.index.is_monotonic_increasing or not train_series.index.is_unique:
        raise ValueError("train_series index must be sorted and unique.")

    forecast_index = pd.DatetimeIndex(forecast_index)
    if forecast_index.empty:
        raise ValueError("forecast_index must contain at least one timestamp.")
    if not forecast_index.is_monotonic_increasing or not forecast_index.is_unique:
        raise ValueError("forecast_index must be sorted and unique.")
    if forecast_index[0] <= train_series.index[-1]:
        raise ValueError("Every forecast timestamp must follow the training period.")
    return forecast_index


### Task 3: implement the benchmark forecast functions

Complete the functions below.

- **Naïve or persistence:** repeat the final observation available at the origin.
- **Daily seasonal naïve:** repeat the final observed 24-hour season.
- **Weekly seasonal naïve:** repeat the final observed 168-hour season.
- **Hybrid weekday/weekend seasonal:** for each target hour, use the latest observation with the same clock hour and the same weekday/weekend class.
- **Trailing moving average:** repeat the mean of the final $k=24$ observations.
- **Simple exponential smoothing:** update the level as

$$
\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}
$$

and repeat the final level. Initialise the level with the first training observation and use $\alpha=0.2$ in the main comparisons.

The seasonal-naïve function must work when the requested horizon exceeds one seasonal period.


In [ ]:
def naive_forecast(train_series, forecast_index):
    '''Repeat the final training observation across the horizon.'''
    forecast_index = _validate_forecast_inputs(train_series, forecast_index)
    # TODO: construct a Series aligned to forecast_index.
    raise NotImplementedError("Implement naive_forecast().")


def seasonal_naive_forecast(train_series, forecast_index, period):
    '''Repeat the final observed season, including for horizons longer than it.'''
    forecast_index = _validate_forecast_inputs(train_series, forecast_index)
    # TODO: extract the last 'period' values and repeat them with np.tile.
    raise NotImplementedError("Implement seasonal_naive_forecast().")


def hybrid_seasonal_forecast(train_series, forecast_index):
    '''Use the latest matching clock hour and weekday/weekend class.'''
    forecast_index = _validate_forecast_inputs(train_series, forecast_index)
    # TODO: for each target timestamp, filter training observations by
    # clock hour and by whether dayofweek >= 5, then take the latest.
    raise NotImplementedError("Implement hybrid_seasonal_forecast().")


def moving_average_forecast(train_series, forecast_index, window=24):
    '''Repeat the final trailing average across the horizon.'''
    forecast_index = _validate_forecast_inputs(train_series, forecast_index)
    # TODO: average only the final 'window' training observations.
    raise NotImplementedError("Implement moving_average_forecast().")


def simple_exponential_smoothing_forecast(
    train_series, forecast_index, alpha=0.2
):
    '''Repeat the final simple-exponential-smoothing level.'''
    forecast_index = _validate_forecast_inputs(train_series, forecast_index)
    # TODO: initialise level with the first observation, apply the
    # lecture recursion, then repeat the final level.
    raise NotImplementedError(
        "Implement simple_exponential_smoothing_forecast()."
    )


### Task 4: implement the forecast metrics

Define the error as $e_t=y_t-\hat{y}_t$ and implement:

$$\mathrm{RMSE}=\sqrt{\mathrm{mean}(e_t^2)}$$

$$\mathrm{MAPE}=100\,\mathrm{mean}\left(\left|\frac{e_t}{y_t}\right|\right)$$

$$\mathrm{MASE}=\frac{\mathrm{mean}(|e_t|)}
{\mathrm{mean}(|y_t-y_{t-m}|)}.$$

For this tutorial, $m=168$. The denominator is the in-sample MAE of a **weekly seasonal-naïve benchmark** on the training series: its forecast for hour $t$ is the observation from hour $t-168$.

The weekly seasonal-naïve forecast is **not forced to have MASE equal to one in a later test period**. Its numerator is its MAE on the test data, whereas the denominator is its MAE on the training data. Its test MASE is exactly one only if those two MAEs happen to be equal. It is exactly one when the benchmark is evaluated on the same training differences used to construct the denominator, as demonstrated by the sanity check below.

- MASE $<1$: lower test MAE than the weekly seasonal-naïve training scale;
- MASE $=1$: test MAE equal to that training scale;
- MASE $>1$: higher test MAE than that training scale.

Check input lengths and pandas index alignment. Reject zero actual values in MAPE and a zero MASE denominator. Always calculate the MASE scale from observations available before the relevant test period; never use the test observations in the denominator.


In [ ]:
def _aligned_finite_arrays(actual, forecast):
    '''Validate two one-dimensional inputs before metric calculation.'''
    # TODO: check lengths and pandas index alignment before calling
    # np.asarray(..., dtype=float). Also reject non-finite values.
    raise NotImplementedError("Complete _aligned_finite_arrays().")


def rmse(actual, forecast):
    '''Root mean squared forecast error.'''
    # TODO: calculate actual - forecast, square, average and take the root.
    raise NotImplementedError("Implement rmse().")


def mape(actual, forecast):
    '''Mean absolute percentage error, returned as a percentage.'''
    # TODO: reject zero actual values explicitly, then return a percentage.
    raise NotImplementedError("Implement mape().")


def mase(actual, forecast, train_series, m=168):
    '''Mean absolute scaled error using a training-only weekly seasonal-naive scale.'''
    # TODO: calculate the numerator on test errors and the denominator
    # from abs(train[m:] - train[:-m]). Check for invalid m and zero scale.
    raise NotImplementedError("Implement mase().")




def score_forecasts(actual, forecast_table, train_series, m=168):
    '''Return RMSE, MAPE and MASE for every forecast column.'''
    # TODO: check index alignment, loop over forecast_table columns,
    # and return a DataFrame with models as rows and metrics as columns.
    raise NotImplementedError("Complete score_forecasts().")


In [ ]:
# Sanity check: score the weekly seasonal-naive benchmark on the same
# training differences used in the MASE denominator.
weekly_benchmark_actual = training_load.iloc[168:]
weekly_benchmark_fitted = pd.Series(
    training_load.iloc[:-168].to_numpy(),
    index=weekly_benchmark_actual.index,
    name="weekly_seasonal_naive_fitted",
)

weekly_benchmark_training_mase = mase(
    weekly_benchmark_actual,
    weekly_benchmark_fitted,
    training_load,
    m=168,
)
print(f"Weekly seasonal-naive MASE on its training scale: {weekly_benchmark_training_mase:.1f}")


## 6. One day-ahead forecast with a fixed 24-hour horizon

The first experiment issues one forecast at 23:00 on 16 December. Every method forecasts the full calendar day of 17 December from this single origin.

### Task 5: construct and evaluate the day-ahead forecasts

Complete `build_benchmark_forecasts` by calling each benchmark with the supplied history and forecast index. Then inspect the metric table and the 24-hour forecast plot.


In [ ]:
def build_benchmark_forecasts(history, forecast_index):
    '''Construct all benchmark forecasts from one fixed origin.'''
    # TODO: create an aligned DataFrame and add all seven benchmark columns.
    # Use only history and forecast_index inside this function.
    raise NotImplementedError("Complete build_benchmark_forecasts().")


day_ahead_forecasts = build_benchmark_forecasts(
    training_load,
    day_ahead_actual.index,
)
day_ahead_scores = score_forecasts(
    day_ahead_actual,
    day_ahead_forecasts,
    training_load,
    m=168,
)
day_ahead_scores.round(3)


In [ ]:
PRINCIPAL_MODELS = [
    "Naïve",
    "Seasonal naïve (24 h)",
    "Seasonal naïve (168 h)",
    "Hybrid weekday/weekend",
]
LEVEL_MODELS = [
    "Historical mean",
    "Moving average (24 h)",
    "SES (alpha=0.2)",
]


def plot_forecast_methods(actual, forecasts, model_names, title, ax=None):
    '''Plot actual demand and selected aligned forecast columns.'''
    if not actual.index.equals(forecasts.index):
        raise ValueError("actual and forecasts must have aligned indexes.")
    if ax is None:
        _, ax = plt.subplots(figsize=(11, 4))

    ax.plot(actual, color="black", linewidth=2.3, label="Actual demand")
    for model_name in model_names:
        ax.plot(forecasts[model_name], linewidth=1.3, label=model_name)
    ax.set(title=title, xlabel="Timestamp", ylabel="Demand (MW)")
    ax.grid(alpha=0.2)
    ax.legend(ncol=2, fontsize=8)
    return ax


fig, ax = plt.subplots(figsize=(11, 4))
plot_forecast_methods(
    day_ahead_actual,
    day_ahead_forecasts,
    list(day_ahead_forecasts.columns),
    "One day-ahead forecast from a fixed origin",
    ax=ax,
)
plt.show()


## 7. Rolling day-ahead forecasts over the complete test set

The rolling experiment starts at 23:00 on 30 September and forecasts every day in the October-to-December test set. After each 24-hour delivery day, the actual observations from that day become available. The origin then moves forward by 24 hours and the next day is forecast from the updated history.

Using an observation after it has occurred is not leakage. A forecast leaks information only when it uses observations that were still in the future at its own origin.

### Task 6: implement the rolling-origin loop

Complete the `for` loop so that it constructs one 24-hour forecast table for every day in `rolling_test_load`. Concatenate the daily tables, evaluate errors over the complete test set, and then visualise the selected December week.


In [ ]:
def rolling_day_ahead_forecasts(complete_series, test_index):
    '''Issue one 24-hour forecast for every complete day in test_index.'''
    test_index = pd.DatetimeIndex(test_index)
    if test_index.empty or len(test_index) % 24 != 0:
        raise ValueError("test_index must contain one or more complete days.")
    if test_index[0].hour != 0 or test_index[-1].hour != 23:
        raise ValueError("test_index must start at 00:00 and end at 23:00.")

    expected_index = pd.date_range(
        test_index[0],
        test_index[-1],
        freq="h",
        name=test_index.name,
    )
    if not test_index.equals(expected_index):
        raise ValueError("test_index must be consecutive, sorted and unique.")

    forecast_parts = []
    origin_parts = []

    # TODO: loop over test_index[::24]. At each day_start:
    # 1. set the origin to one hour before day_start;
    # 2. slice the complete history through that origin;
    # 3. create the following 24-hour forecast index;
    # 4. call build_benchmark_forecasts and store the result and origin.
    raise NotImplementedError("Complete the rolling day-ahead for loop.")


rolling_forecasts, rolling_origin_by_timestamp = rolling_day_ahead_forecasts(
    aggregate_load_2007,
    rolling_test_load.index,
)
rolling_test_scores = score_forecasts(
    rolling_test_load,
    rolling_forecasts,
    rolling_training_load,
    m=168,
)
rolling_test_scores.round(3)


In [ ]:
# Select one week for a readable illustration of the longer rolling test.
rolling_week_forecasts = rolling_forecasts.loc[test_week.index]

fig, ax = plt.subplots(figsize=(12, 5))
plot_forecast_methods(
    test_week,
    rolling_week_forecasts,
    PRINCIPAL_MODELS,
    "Rolling day-ahead forecasts: one week from the complete test set",
    ax=ax,
)
for day_start in pd.date_range(test_week.index[0], periods=7, freq="24h"):
    ax.axvline(day_start, color="0.5", linestyle=":", linewidth=0.8, alpha=0.6)
plt.show()


## 8. Fixed-horizon forecast for the full week

The final experiment returns to the origin at 23:00 on 16 December and requests all 168 hours at once. No observation from the illustration week can update this forecast.

### Task 7: construct and evaluate the fixed week-ahead forecasts

Use `build_benchmark_forecasts` with the fixed-origin training data and the full illustration-week index. Compare its metric table with the rolling forecasts for the same week.


In [ ]:
# TODO: build the 168-hour forecast from the initial training period,
# then calculate its comparison table.
raise NotImplementedError("Construct and evaluate fixed_week_forecasts.")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
plot_forecast_methods(
    test_week,
    fixed_week_forecasts,
    PRINCIPAL_MODELS,
    "Fixed week-ahead seasonal benchmarks",
    ax=axes[0],
)
plot_forecast_methods(
    test_week,
    fixed_week_forecasts,
    LEVEL_MODELS,
    "Fixed week-ahead level-based benchmarks",
    ax=axes[1],
)
plt.show()


### Rolling versus fixed-week performance

The rolling metric table above covers the complete October-to-December test set. For a like-for-like comparison here, we select the rolling forecasts for 17–23 December and score them against the same 168 observations as the fixed week-ahead forecast. Both tables use the weekly seasonal-naïve MASE scale calculated from `training_load`; their structural difference is whether the information set updates every 24 hours.


In [ ]:
rolling_week_scores = score_forecasts(
    test_week,
    rolling_week_forecasts,
    training_load,
    m=168,
)

rolling_fixed_comparison = pd.concat(
    {
        "Rolling day-ahead": rolling_week_scores,
        "Fixed week-ahead": fixed_week_scores,
    },
    axis=1,
)
display(rolling_fixed_comparison.round(3))

rmse_comparison = pd.DataFrame(
    {
        "Rolling day-ahead": rolling_week_scores["RMSE (MW)"],
        "Fixed week-ahead": fixed_week_scores["RMSE (MW)"],
    }
)
ax = rmse_comparison.plot(kind="bar", figsize=(11, 4), width=0.8)
ax.set(
    title="RMSE for rolling and fixed-week forecasts",
    xlabel="Forecast method",
    ylabel="RMSE (MW)",
)
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=35, ha="right")
plt.show()


## 9. Interpretation questions

1. Which method performs best for the first 24-hour day-ahead forecast under each metric?
2. What new information becomes available at each rolling origin, and why is using it not data leakage?
3. Which methods improve most when the origin is updated daily rather than fixed for the illustration week?
4. Do RMSE, MAPE and MASE produce the same model ranking in every experiment?
5. Which forecasts represent daily seasonality, weekly seasonality, or the weekday/weekend distinction?
6. Why do the moving-average and simple exponential-smoothing forecasts remain flat within each multi-step horizon?
7. Which benchmark defines the MASE denominator here, and what does a MASE below one mean?
8. How might holidays, changing weather or unusual building operation affect the seasonal benchmarks?
9. How would an operational origin around 11:00–12:00 on day $D-1$ change the information available for forecasting day $D$?


### Your answers

1. _TODO_
2. _TODO_
3. _TODO_
4. _TODO_
5. _TODO_
6. _TODO_
7. _TODO_
8. _TODO_
9. _TODO_
